# Notebook 2 — Muestreo espacial: ¿por qué la imagen pixelada pierde detalle?

**muestreo (espacial)** y el criterio tipo Nyquist–Shannon. En imágenes, esto se interpreta como:
- Si el detalle de la escena cambia muy rápido (muchas “rayitas” o texturas finas) y muestreamos con pocos píxeles,
  aparece **aliasing** (patrones falsos, moiré).

Ejemplo cotidiano e industrial: **fotografiar una camisa con rayas** o una **malla metálica**.


In [ ]:
# (1) Importamos librerías/módulos que vamos a usar.
import numpy as np
# (2) Importamos librerías/módulos que vamos a usar.
import matplotlib.pyplot as plt

## A) Generar un patrón de “rayas” (alta frecuencia)
Usaremos una señal 2D con rayas. Luego la muestreamos a menor resolución.


In [ ]:
# 1) Creamos una imagen con rayas verticales: alternancia rápida.
# (1) Ejecutamos esta instrucción como parte del procedimiento.
H, W = 300, 300
# (2) Definimos/asignamos la variable `x`.
x = np.arange(W)[None, :]

# 2) Frecuencia de rayas: cada k píxeles cambia de blanco a negro.
# (3) Definimos/asignamos la variable `k`.
k = 6
# (4) Definimos/asignamos la variable `patron`.
patron = ((x // k) % 2) * 255
# (5) Definimos/asignamos la variable `patron`.
patron = np.repeat(patron, H, axis=0).astype(np.uint8)

# (6) Creamos una nueva figura para graficar.
plt.figure(figsize=(5, 5))
# (7) Mostramos una matriz como imagen.
plt.imshow(patron, cmap='gray', vmin=0, vmax=255)
# (8) Agregamos un título a la gráfica.
plt.title('Patrón original (alta frecuencia espacial)')
# (9) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')
# (10) Renderizamos las gráficas en pantalla.
plt.show()

## B) Submuestreo: quedarnos con menos píxeles
Submuestrear es como tomar “una de cada N filas/columnas”.
Esto se parece a usar una cámara con menos resolución o alejarte mucho.


In [ ]:
# 3) Función simple de submuestreo (sin filtro anti-alias).
# (1) Definimos la función `submuestrear`.
def submuestrear(im, factor):
    # (2) Regresamos el resultado de la función.
    return im[::factor, ::factor]

# 4) Probamos diferentes factores.
# (3) Definimos/asignamos la variable `im2`.
im2 = submuestrear(patron, 2)
# (4) Definimos/asignamos la variable `im5`.
im5 = submuestrear(patron, 5)
# (5) Definimos/asignamos la variable `im10`.
im10 = submuestrear(patron, 10)

# (6) Creamos una nueva figura para graficar.
plt.figure(figsize=(12, 4))
# (7) Iniciamos un ciclo para repetir acciones.
for i, (img, t) in enumerate([(im2,'factor 2'), (im5,'factor 5'), (im10,'factor 10')], start=1):
    # (8) Seleccionamos una zona (subplot) dentro de la figura.
    plt.subplot(1, 3, i)
    # (9) Mostramos una matriz como imagen.
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    # (10) Agregamos un título a la gráfica.
    plt.title(f'Submuestreo {t}')
    # (11) Configuramos/ocultamos ejes para visualizar mejor.
    plt.axis('off')
# (12) Renderizamos las gráficas en pantalla.
plt.show()

### ¿Por qué se ve raro?
Porque tomamos pocas muestras para una señal que cambia rápido. En términos del PDF:
- Necesitamos que la frecuencia de muestreo sea suficientemente alta para capturar el detalle.


## C) Solución práctica: filtro anti-alias (promediado) antes de submuestrear
En industria, antes de reducir resolución se filtra (suaviza) para evitar aliasing.
Aquí haremos un filtro muy simple: promedio por bloques.


In [ ]:
# 5) Promedio por bloques (box filter) + decimación.
# (1) Definimos la función `downsample_promedio`.
def downsample_promedio(im, factor):
    # (2) Ejecutamos esta instrucción como parte del procedimiento.
    H, W = im.shape
    # (3) Definimos/asignamos la variable `H2`.
    H2 = (H // factor) * factor
    # (4) Definimos/asignamos la variable `W2`.
    W2 = (W // factor) * factor
    # (5) Definimos/asignamos la variable `imc`.
    imc = im[:H2, :W2].astype(np.float32)
    # Reorganizamos en bloques (factor x factor) y promediamos.
    # (6) Definimos/asignamos la variable `imc`.
    imc = imc.reshape(H2//factor, factor, W2//factor, factor).mean(axis=(1,3))
    # (7) Convertimos el tipo de dato (por ejemplo a float o uint8).
    return imc.astype(np.uint8)

# (8) Definimos/asignamos la variable `im10_aa`.
im10_aa = downsample_promedio(patron, 10)

# (9) Creamos una nueva figura para graficar.
plt.figure(figsize=(10,4))
# (10) Seleccionamos una zona (subplot) dentro de la figura.
plt.subplot(1,2,1)
# (11) Mostramos una matriz como imagen.
plt.imshow(im10, cmap='gray', vmin=0, vmax=255)
# (12) Agregamos un título a la gráfica.
plt.title('Submuestreo factor 10 (sin anti-alias)')
# (13) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')

# (14) Seleccionamos una zona (subplot) dentro de la figura.
plt.subplot(1,2,2)
# (15) Mostramos una matriz como imagen.
plt.imshow(im10_aa, cmap='gray', vmin=0, vmax=255)
# (16) Agregamos un título a la gráfica.
plt.title('Downsample factor 10 (con promedio)')
# (17) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')
# (18) Renderizamos las gráficas en pantalla.
plt.show()

### Conexión industria
- Inspección de **PCB** o etiquetas con patrones finos: si la cámara o el lente no dan suficiente resolución, pueden aparecer falsos patrones.
- Por eso, elegir resolución (espacial) y distancia focal no es “lujo”, es requisito de calidad.
